[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/efficient/examples/comparative.ipynb)

# Static MISDA and PCA comparison

This notebook compares MISDA and PCA as reduced representations of the same original objective space. MISDA preserves selected original objectives; PCA uses transformed principal components. Their native diagnostics remain separate, while direct comparison uses a common leave-one-out reconstruction score at the same reduced dimension.

In [ ]:
# Install MISDA with optional benchmarks suite from the efficient branch
!pip install --quiet --upgrade "git+https://github.com/monacofj/misda.git@efficient#egg=misda[benchmarks]"


In [ ]:
import pandas as pd

import misda
import misda.benchmarks as bench

N = 500
SEED = 123

COMPARATIVE_CASES = (
    ("exp_01", bench.mopA_monotonic_redundancy),
    ("exp_02", bench.mopC_latent_blocks_4x5),
    ("exp_03", bench.mopD_pure_conflict_groups),
)

## Run the three comparative experiments

Each experiment uses the public MISDA API directly, exactly as the other example notebooks do. The common score is `global_standardized_external_r2`: each non-constant original objective has equal weight, and reconstruction is evaluated leave-one-out. Preserved MISDA objectives contribute exact reconstruction (`R²=1`); eliminated objectives contribute MISDA's stored external PRESS/LOO R². PCA centering, scaling and principal directions are refit without the held-out row.

In [ ]:
comparative_results = {}
comparison_rows = []

for case_id, generator in COMPARATIVE_CASES:
    data, truth = generator(N=N, seed=SEED)
    result = misda.analyze(
        data,
        method="static",
        name=truth["name"],
        seed=SEED,
        max_evaluated_mis=1,
    )
    benchmark_result = misda.benchmark(result, truth)
    print(benchmark_result.report())
    result.graph_plot()

    misda_common = bench.misda_global_standardized_external_r2(data, result)
    pca_external_curve = bench.pca_external_reconstruction_curve(
        data, max_components=data.shape[1]
    )
    selected_dimension = result.selected_dimension
    pca_same_dimension = next(
        point[bench.COMMON_RECONSTRUCTION_METRIC]
        for point in pca_external_curve
        if point["dimension"] == selected_dimension
    )
    pca_native_curve = bench.pca_in_sample_reconstruction_curve(
        data, max_components=min(10, data.shape[1])
    )

    comparative_results[case_id] = {
        "result_obj": result,
        "benchmark_obj": benchmark_result,
        "truth": truth,
        "misda_common": misda_common,
        "pca_external_curve": pca_external_curve,
        "pca_native_curve": pca_native_curve,
    }
    comparison_rows.append(
        {
            "case_id": case_id,
            "name": truth["name"],
            "dimension": selected_dimension,
            "misda_global_standardized_external_r2": misda_common,
            "pca_global_standardized_external_r2": pca_same_dimension,
            "misda_minus_pca": misda_common - pca_same_dimension,
        }
    )

## Direct comparison at the MISDA-selected dimension

This is the only table intended for a numerical MISDA-versus-PCA comparison. Both columns use the same external, equal-objective-weight reconstruction estimand. A higher value means that the reduced representation reconstructs more of the standardized original objective space.

In [ ]:
comparison = pd.DataFrame(comparison_rows)
comparison

## MISDA native reconstruction diagnostics

These remain MISDA-specific: they summarize only objectives eliminated by the preferred MIS. They are useful for diagnosing the reduction but are not directly compared with PCA's native score.

In [ ]:
misda_reconstruction = pd.DataFrame(
    {
        "case_id": case_id,
        "name": item["truth"]["name"],
        "selected_dimension": item["result_obj"].selected_dimension,
        "mean_eliminated_objective_r2": item["result_obj"].best_mis.evaluation["linear_reconstruction"]["mean_r2"],
        "worst_eliminated_objective_r2": item["result_obj"].best_mis.evaluation["linear_reconstruction"]["worst_r2"],
    }
    for case_id, item in comparative_results.items()
)
misda_reconstruction

## PCA native in-sample reconstruction

For continuity with the previous comparative artifact, the conventional PCA `global_standardized_r2` curve is retained as a separate in-sample diagnostic. It must not be compared numerically with MISDA's eliminated-objective R².

In [ ]:
pca_reconstruction = pd.DataFrame(
    {
        "case_id": case_id,
        "name": item["truth"]["name"],
        "dimension": point["dimension"],
        "global_standardized_r2": point["global_standardized_r2"],
    }
    for case_id, item in comparative_results.items()
    for point in item["pca_native_curve"]
)
pca_reconstruction